In [82]:
import numpy as np
import plotly.graph_objects as go

SEED = 2509

def draw_payoffs(n_rounds, rng, p=0.5, base=2.0):
    n_flips = rng.geometric(p, size=n_rounds)
    return base ** n_flips.astype(np.float64)

In [83]:
def running_mean(x):
    return np.cumsum(x) / np.arange(1, x.size + 1)

In [84]:
def mc_ensemble_means(n_grid, n_replications=200, seed=SEED, p=0.5, base=2.0):
    n_grid = np.asarray(n_grid, dtype=np.int64)
    n_max = int(n_grid.max())
    out = np.empty((n_replications, n_grid.size))
    rng = np.random.default_rng(seed)
    for r in range(n_replications):
        payoffs = draw_payoffs(n_max, rng, p=p, base=base)
        out[r] = running_mean(payoffs)[n_grid - 1]
    return out

In [ ]:
import plotly.graph_objects as go

def plot_ensemble_means(n_grid, means, quantiles=(0.05, 0.25, 0.75, 0.95)):
    n_grid = np.asarray(n_grid)
    q = np.quantile(means, quantiles, axis=0)
    median = np.median(means, axis=0)

    fig = go.Figure()
    # 5-95 band
    fig.add_trace(go.Scatter(x=np.concatenate([n_grid, n_grid[::-1]]),
                             y=np.concatenate([q[3], q[0][::-1]]),
                             fill="toself", fillcolor="rgba(70,110,170,0.15)",
                             line=dict(width=0), hoverinfo="skip",
                             name="5–95%"))
    # 25-75 band
    fig.add_trace(go.Scatter(x=np.concatenate([n_grid, n_grid[::-1]]),
                             y=np.concatenate([q[2], q[1][::-1]]),
                             fill="toself", fillcolor="rgba(70,110,170,0.30)",
                             line=dict(width=0), hoverinfo="skip",
                             name="25–75%"))
    fig.add_trace(go.Scatter(x=n_grid, y=median, mode="lines+markers",
                             line=dict(color="rgb(30,60,120)", width=2),
                             name="median"))

    fig.update_layout(
        xaxis=dict(type="log", title="sample size N"),
        yaxis=dict(title="sample mean payoff"),
        template="simple_white",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    return fig


In [106]:
n_grid = np.unique(np.logspace(0, 6, 30).astype(np.int64))
means = mc_ensemble_means(n_grid, n_replications=2000)

In [107]:
plot_ensemble_means(n_grid, means).show()